# Tutorial 1: Causal Patching & Metrics

This notebook teaches you how to use mlxterp's causal patching tools to identify which model components are responsible for specific behaviors.

**What you'll learn:**
- How activation patching works and when to use it
- Layer-level, position-level, and head-level patching
- Using different metrics (l2, logit_diff, kl, cosine)
- The `CausalTrace` context manager for multi-patch experiments
- Working with `PatchingResult` objects

**Prerequisites:** mlxterp installed with `pip install -e ".[all]"`

## 1. Setup

Load a model and verify everything works.

In [ ]:
from mlxterp import InterpretableModel
from mlxterp.causal import activation_patching, CausalTrace
from mlxterp.metrics import logit_diff, kl_divergence, l2_distance, cosine_distance, get_metric
import mlx.core as mx

# Load a model — this downloads ~700MB on first run
model = InterpretableModel("mlx-community/Llama-3.2-1B-Instruct-4bit")
print(f"Model loaded: {len(model.layers)} layers")
print(f"Tokenizer: {type(model.tokenizer).__name__}")

## 2. Understanding Activation Patching

**The core idea:** Given a clean input (correct behavior) and a corrupted input (wrong behavior), we can identify which components cause the difference by swapping activations between the two runs.

If patching layer 5's MLP from clean into corrupted *recovers* the clean output, then layer 5's MLP is causally important for this behavior.

In [ ]:
# Define our experiment
clean = "The Eiffel Tower is in"
corrupted = "The Colosseum is in"

# What does the model predict for each?
with model.trace(clean) as trace:
    clean_output = model.output.save()

with model.trace(corrupted) as trace:
    corrupted_output = model.output.save()

mx.eval(clean_output, corrupted_output)

# Top predictions
clean_top = model.tokenizer.decode([int(mx.argmax(clean_output[0, -1, :]))])
corrupted_top = model.tokenizer.decode([int(mx.argmax(corrupted_output[0, -1, :]))])

print(f"Clean: '{clean}' → predicts '{clean_top}'")
print(f"Corrupted: '{corrupted}' → predicts '{corrupted_top}'")

## 3. Layer-Level Patching

Which MLPs and attention layers matter for this factual recall?

In [ ]:
# Patch MLPs at each layer
mlp_result = activation_patching(
    model,
    clean=clean,
    corrupted=corrupted,
    component="mlp",
    metric="l2",
)

print("MLP patching results:")
print(mlp_result.summary())
print()

# Top 5 most important layers
print("Top 5 important MLP layers:")
for layer_idx, effect in mlp_result.top_components(k=5):
    print(f"  Layer {layer_idx}: effect = {effect:.4f}")

In [ ]:
# Visualize the results
mlp_result.plot()

In [ ]:
# Now do the same for attention
attn_result = activation_patching(
    model,
    clean=clean,
    corrupted=corrupted,
    component="attn",
    metric="l2",
)

print("Attention patching results:")
print(attn_result.summary())
print()

print("Top 5 important attention layers:")
for layer_idx, effect in attn_result.top_components(k=5):
    print(f"  Layer {layer_idx}: effect = {effect:.4f}")

attn_result.plot()

## 4. Comparing Metrics

Different metrics capture different aspects of the patching effect. Let's compare them.

In [ ]:
# Compare l2 vs cosine vs kl metrics
from mlxterp.visualization.patching import plot_patching_comparison

results = []
for metric_name in ["l2", "cosine", "kl"]:
    r = activation_patching(
        model, clean, corrupted,
        component="mlp",
        metric=metric_name,
    )
    results.append(r)
    print(f"{metric_name}: {r.summary()}")

plot_patching_comparison(results, title="MLP Patching: Metric Comparison")

## 5. Using logit_diff Metric

For tasks with a clear correct vs incorrect answer, `logit_diff` is the most precise metric.

In [ ]:
# Find token IDs for the correct and incorrect answers
correct_token = model.tokenizer.encode(" Paris")[-1]
incorrect_token = model.tokenizer.encode(" Rome")[-1]

print(f"Correct token: '{model.tokenizer.decode([correct_token])}' (id={correct_token})")
print(f"Incorrect token: '{model.tokenizer.decode([incorrect_token])}' (id={incorrect_token})")

# Run patching with logit_diff
ld_result = activation_patching(
    model, clean, corrupted,
    component="mlp",
    metric="logit_diff",
    metric_kwargs={
        "correct_token": correct_token,
        "incorrect_token": incorrect_token,
    },
)

print("\nlogit_diff results:")
for layer_idx, effect in ld_result.top_components(k=5):
    print(f"  Layer {layer_idx}: recovery = {effect:.1%}")

ld_result.plot()

## 6. Position-Level Patching

Which token positions carry the crucial information?

In [ ]:
# Show the tokens in our prompt
clean_tokens = model.tokenizer.encode(clean)
print("Token positions:")
for i, tid in enumerate(clean_tokens):
    print(f"  Position {i}: '{model.tokenizer.decode([tid])}'")

In [ ]:
# Patch only the subject token positions
# (e.g., positions where "Eiffel Tower" vs "Colosseum" differ)
# You may need to adjust these based on the tokenization above
subject_positions = [1, 2, 3]  # Adjust based on your tokenizer output

pos_result = activation_patching(
    model, clean, corrupted,
    component="mlp",
    metric="l2",
    positions=subject_positions,
)

print(f"Position-level patching (positions {subject_positions}):")
for layer_idx, effect in pos_result.top_components(k=5):
    print(f"  Layer {layer_idx}: effect = {effect:.4f}")

## 7. CausalTrace: Multi-Patch Experiments

When you want to patch multiple components simultaneously, use `CausalTrace` instead of running separate experiments.

In [ ]:
# Patch multiple components at once
with model.causal_trace(clean, corrupted) as ct:
    # How much do the clean activations available?
    print(f"Clean activations captured: {len(ct.clean_activations)} keys")
    print(f"Clean output shape: {ct.clean_output.shape}")
    print(f"Corrupted output shape: {ct.corrupted_output.shape}")

In [ ]:
# Patch several layers and measure combined effect
with model.causal_trace(clean, corrupted) as ct:
    # Patch MLPs at the most important layers
    top_layers = [l for l, _ in mlp_result.top_components(k=3)]
    for layer_idx in top_layers:
        ct.patch(f"layers.{layer_idx}.mlp")
    
    effect = ct.metric("l2")
    print(f"Patching layers {top_layers} together:")
    print(f"  Combined l2 recovery: {effect:.4f}")

In [ ]:
# Use canonical names with layer parameter
with model.causal_trace(clean, corrupted) as ct:
    ct.patch("mlp", layer=5)
    ct.patch("attn", layer=5)  # Patch both MLP and attention at layer 5
    
    effect = ct.metric("l2")
    print(f"Layer 5 (MLP + Attention): l2 recovery = {effect:.4f}")

In [ ]:
# Access clean activations directly
with model.causal_trace(clean, corrupted) as ct:
    mlp5_act = ct.get_clean_activation("mlp", layer=5)
    print(f"Clean MLP layer 5 activation shape: {mlp5_act.shape}")
    print(f"Mean activation: {float(mx.mean(mlp5_act)):.4f}")
    print(f"Max activation: {float(mx.max(mlp5_act)):.4f}")

## 8. Working with Results

Every patching function returns a `PatchingResult` with rich output options.

In [ ]:
import json

# Summary
print("Summary:", mlp_result.summary())
print()

# JSON export (useful for programmatic analysis or agent consumption)
json_data = json.loads(mlp_result.to_json())
print("JSON keys:", list(json_data.keys()))
print(f"Result type: {json_data['result_type']}")
print()

# Markdown report
print(mlp_result.to_markdown())

In [ ]:
# Raw effect matrix
print(f"Effect matrix shape: {mlp_result.effect_matrix.shape}")
print(f"Effect matrix: {mlp_result.effect_matrix.tolist()}")
print(f"\nLayers tested: {mlp_result.layers}")
print(f"Component: {mlp_result.component}")
print(f"Metric: {mlp_result.metric_name}")

## 9. Custom Metrics

You can define your own metric function and pass it to any patching tool.

In [ ]:
# Custom metric: probability of the correct token
def prob_recovery(patched_logits, clean_logits, corrupted_logits, target_token=None, **kwargs):
    """Measure how much the correct token's probability is recovered."""
    if target_token is None:
        target_token = int(mx.argmax(clean_logits[0]))
    
    clean_prob = float(mx.softmax(clean_logits[0])[target_token])
    corrupted_prob = float(mx.softmax(corrupted_logits[0])[target_token])
    patched_prob = float(mx.softmax(patched_logits[0])[target_token])
    
    if abs(clean_prob - corrupted_prob) < 1e-10:
        return 0.0
    return (patched_prob - corrupted_prob) / (clean_prob - corrupted_prob)

# Use it!
custom_result = activation_patching(
    model, clean, corrupted,
    component="mlp",
    metric=prob_recovery,
)

print("Custom metric results:")
for layer_idx, effect in custom_result.top_components(k=5):
    print(f"  Layer {layer_idx}: probability recovery = {effect:.1%}")

## Summary

In this tutorial you learned:

| Tool | Purpose | When to Use |
|------|---------|-------------|
| `activation_patching()` | Identify important components | Start of any causal analysis |
| `component="mlp"` / `"attn"` | Layer-level analysis | Finding which layers matter |
| `component="attn_head"` | Head-level analysis | Drilling into specific heads |
| `positions=[...]` | Position-level analysis | Finding which tokens matter |
| `CausalTrace` | Multi-patch experiments | Combining multiple patches |
| `metric="logit_diff"` | Precise token-level metric | Tasks with clear correct/incorrect answers |
| `result.plot()` | Visualization | Quick visual exploration |
| `result.to_json()` | Export | Programmatic processing |

**Next:** Try [02_circuit_discovery.ipynb](02_circuit_discovery.ipynb) to learn DLA, path patching, and ACDC.